In [ ]:
!pip install -q unsloth
# Also get the latest nightly Unsloth!
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [ ]:
import json

input_file = "karnataka_schemes_sft_dataset.jsonl"
output_file = "data_with_answers.json"

data = []

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Conversion completed!")

Conversion completed!


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import torch

# ── Hyperparameters ──────────────────────────────────────────
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH  = 2048   # Unsloth handles RoPE scaling internally
DTYPE           = None   # Auto-detect: float16 (T4/V100), bfloat16 (Ampere+)
LOAD_IN_4BIT    = True   # 4-bit QLoRA — saves ~60 % VRAM

# LoRA rank: 16 is a good default. Use 32/64 for harder tasks.
LORA_R          = 16
LORA_ALPHA      = 32     # Alpha = 2 × r is a reliable default (scales LoRA updates)

# Training
BATCH_SIZE              = 2
GRAD_ACCUM_STEPS        = 4   # Effective batch = 2 × 4 = 8
NUM_TRAIN_EPOCHS        = 3   # 4309 samples / 8 eff-batch ≈ 539 steps/epoch → ~1617 total
WARMUP_STEPS            = 50  # ≈ first 10 % of epoch 1
LEARNING_RATE           = 2e-4
LR_SCHEDULER            = "cosine"   # Cosine decay generalises better than linear
WEIGHT_DECAY            = 0.01
LOGGING_STEPS           = 25   # Log every 25 steps (not every 1 — avoids noisy console)
SAVE_STEPS              = 200  # Save checkpoint every 200 steps
OUTPUT_DIR              = "outputs"

# Inference (after training)
INFERENCE_TEMPERATURE   = 0.3   # Low temp → factual, deterministic answers
INFERENCE_MAX_NEW_TOKENS = 512

# Save paths
LORA_SAVE_PATH   = "Karnataka-Llama-3.2-3B-LoRA"      # LoRA adapters only
MERGED_SAVE_PATH = "Karnataka-Llama-3.2-3B-Merged"    # Full merged model (fp16)
GGUF_SAVE_PATH   = "Karnataka-Llama-3.2-3B-GGUF"      # GGUF for llama.cpp / Ollama



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r               = LORA_R,
    target_modules  = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = 0,          # 0 is optimised in Unsloth
    bias            = "none",
    use_gradient_checkpointing = "unsloth",  # Saves VRAM for long sequences
    random_state    = 3407,
    use_rslora      = False,
    loftq_config    = None,
)

print(model.print_trainable_parameters())

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
None


In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# System message that will be prepended to every training example.
# This teaches the model its role during fine-tuning — critical for
# domain adaptation.
SYSTEM_MESSAGE = (
    "You are a knowledgeable and helpful Karnataka government schemes assistant. "
    "Answer citizen queries about Karnataka state government schemes accurately, "
    "concisely, and in a friendly tone. Base your answers strictly on factual "
    "information about eligibility, benefits, application process, required "
    "documents, and other scheme details. Do not hallucinate information."
)

def format_to_chat(examples):
    """
    Converts each {instruction, input, output} record into a full
    Llama-3.1 chat-template string.

    Training loss is computed ONLY on the assistant response tokens
    (handled later by train_on_responses_only). The prompt tokens are
    masked to zero loss — the model learns to answer, not to parrot
    the question back.
    """
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]

    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        # Build user turn: combine instruction + optional context input
        user_content = instruction.strip()
        if inp and inp.strip():
            user_content += f"\n\nContext: {inp.strip()}"

        messages = [
            {"role": "system",    "content": SYSTEM_MESSAGE},
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": output.strip()},
        ]

        # apply_chat_template adds all special tokens; add_generation_prompt
        # must be False here because the assistant response is already included.
        text = tokenizer.apply_chat_template(
            messages,
            tokenize            = False,
            add_generation_prompt = False,
        )
        texts.append(text)

    return {"text": texts}


# Load directly from JSONL — no intermediate JSON conversion needed.
# Each row: {"instruction": "...", "input": "", "output": "..."}
dataset = load_dataset(
    "json",
    data_files = "karnataka_schemes_sft_dataset.jsonl",
    split      = "train",
)

print(f"Dataset loaded: {len(dataset)} records")
print("Sample record:")
print(dataset[0])

# Apply chat template formatting
dataset = dataset.map(format_to_chat, batched=True)

# Quick sanity check — print one formatted training example
print("\n── Formatted training example (first record) ──")
print(dataset[0]["text"][:800])
print("...")


Dataset loaded: 4309 records
Sample record:
{'instruction': 'What is the Krushi Bhagya Scheme?', 'input': '', 'output': 'Krushi Bhagya Scheme is a Karnataka government scheme. Category: Irrigation. Managing Department: Department of Agriculture, karnataka. Key Benefit: Subsidy up to 90% on farm pond construction and irrigation infrastructure. Level: State. Target Group: Farmers owning 1 acre or more land; valid farmer ID.'}

── Formatted training example (first record) ──
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a knowledgeable and helpful Karnataka government schemes assistant. Answer citizen queries about Karnataka state government schemes accurately, concisely, and in a friendly tone. Base your answers strictly on factual information about eligibility, benefits, application process, required documents, and other scheme details. Do not hallucinate information.<|eot_id|><|start_header_id|>user<

In [ ]:
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,
        num_train_epochs            = NUM_TRAIN_EPOCHS,
        warmup_steps                = WARMUP_STEPS,
        learning_rate               = LEARNING_RATE,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = LOGGING_STEPS,
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 2,          # Keep only last 2 checkpoints
        optim                       = "adamw_8bit",
        weight_decay                = WEIGHT_DECAY,
        lr_scheduler_type           = LR_SCHEDULER,
        seed                        = 3407,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",     # Set "wandb" to enable W&B logging
    ),
)

# ── Response-only loss masking ────────────────────────────────
# This is the KEY fix: loss is computed ONLY on assistant response
# tokens, not on the system prompt or user question tokens.
# Without this, the model wastes capacity memorising the prompt.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4309 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/4309 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/4309 [00:00<?, ? examples/s]

In [ ]:
model.config.use_cache = False

print("Starting training...")
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"  Total steps   : {trainer_stats.global_step}")
print(f"  Training loss : {trainer_stats.training_loss:.4f}")
print(f"  Runtime       : {trainer_stats.metrics['train_runtime']:.0f}s")


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,309 | Num Epochs = 3 | Total steps = 1,617
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
25,0.153265
50,0.111105
75,0.119149
100,0.123697
125,0.117501
150,0.119401
175,0.118435
200,0.096744
225,0.097618
250,0.092909


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1617/tokenizer_config.json.



Training complete!
  Total steps   : 1617
  Training loss : 0.0372
  Runtime       : 3363s


In [ ]:
FastLanguageModel.for_inference(model)

def ask_scheme_assistant(question: str) -> str:
    """Run a single question through the fine-tuned model."""
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,   # True at inference — model completes the answer
        return_tensors        = "pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids        = inputs,
            max_new_tokens   = INFERENCE_MAX_NEW_TOKENS,
            use_cache        = True,
            temperature      = INFERENCE_TEMPERATURE,
            do_sample        = True,
            repetition_penalty = 1.1,
        )

    # Decode only the newly generated tokens (skip the prompt)
    new_tokens = outputs[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Test with a few Karnataka scheme questions
test_questions = [
    "What is the Gruha Lakshmi Scheme and who is eligible?",
    "What documents do I need to apply for the Yuva Nidhi Scheme?",
    "How can I apply for the Shakti Scheme for free bus travel?",
    "What benefits does the Bhagyalakshmi Scheme offer for girl children?",
    "Anna Bhagya Scheme inda yenu sige? (What do we get from Anna Bhagya Scheme?)",
]

print("\n" + "=" * 60)
print("INFERENCE TESTS")
print("=" * 60)
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask_scheme_assistant(q)}")
    print("-" * 60)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



INFERENCE TESTS

Q: What is the Gruha Lakshmi Scheme and who is eligible?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

A: Gruha Lakshmi Scheme is a Karnataka government scheme. Eligibility: Women heads of households from Below Poverty Line (BPL) families in Karnataka

Age 18 years and above

Resident of Karnataka
------------------------------------------------------------

Q: What documents do I need to apply for the Yuva Nidhi Scheme?


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: To apply for Yuva Nidhi Scheme, you will need the following documents: Aadhaar Card

University Seat Number or Diploma Register Number

CET/Ration card, caste/income/residence certificate (for domicile verification)

Degree or Diploma certificate

Mobile number and email ID

Bank account details

Self-declaration and signature
------------------------------------------------------------

Q: How can I apply for the Shakti Scheme for free bus travel?


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Here is how you can apply for Shakti Scheme for free bus travel: Online:

Visit Seva Sindhu: https://sevasindhuservices.karnataka.gov.in/

Register on portal and click “Apply for Shakti Smart Card”

Fill the online application form and upload ID proof

Application is verified by the department; Shakti Smart Card is issued within 90 days

Until card is received, show any Karnataka-based photo ID on Karnataka-run (KSRTC/NEKRTC/NWKRTC/BMTC) buses for free travel

Offline:

For immediate benefit before Smart Card: Show valid photo ID to bus staff; travel is free

No paper form for application; all permanent enrollment is digital

Sample application screenshots available online
------------------------------------------------------------

Q: What benefits does the Bhagyalakshmi Scheme offer for girl children?


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: The Bhagyalakshmi Scheme provides the following benefits for girl children: Financial assistance to support girl child upbringing
------------------------------------------------------------

Q: Anna Bhagya Scheme inda yenu sige? (What do we get from Anna Bhagya Scheme?)
A: Under the Anna Bhagya Scheme, you can receive: 10 kg free rice per person per month to BPL families (5 kg from Centre under NFSA + 5 kg from Karnataka govt); OR equivalent cash transfer via DBT; Recently modified to include Indira Nutrition Kit (toor dal, moong dal, sugar, salt, cooking oil) for 5 kg rice portion
------------------------------------------------------------


In [ ]:
model.save_pretrained_merged(
    MERGED_SAVE_PATH,
    tokenizer,
    save_method = "merged_16bit",   # Options: "merged_16bit", "merged_4bit", "lora"
)
print(f"Merged float16 model saved to: {MERGED_SAVE_PATH}/")


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in Karnataka-Llama-3.2-3B-Merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:19<00:00, 39.57s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:30<00:00, 45.40s/it]


Unsloth: Merge process complete. Saved to `/content/Karnataka-Llama-3.2-3B-Merged`
Merged float16 model saved to: Karnataka-Llama-3.2-3B-Merged/


In [ ]:
model.save_pretrained_gguf(
    GGUF_SAVE_PATH,
    tokenizer,
    quantization_method = "q4_k_m",
)
print(f"GGUF model saved to: {GGUF_SAVE_PATH}/")


Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in Karnataka-Llama-3.2-3B-GGUF/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:11<00:00, 65.79s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:15<00:00, 67.72s/it]


Unsloth: Merge process complete. Saved to `/content/Karnataka-Llama-3.2-3B-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Karnataka-Llama-3.2-3B-GGUF_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['Karnataka-Llama-3.2-3B-GGUF_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model Karnataka-Llama-3.2-3B-GGUF_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to Karnataka-Llama-3.2-3B-GGUF_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f Karnataka-Llama-3.2-3B-GGUF_gguf/Modelfile
GGUF model saved to: Karnataka-Llama-3.2-3B-GGUF/
